# Day 4 — Hyperparameter Tuning & Overfitting Analysis

**Objective:** Optimize the top 2 candidate models from Day 3 systematically,
keep a full experiment history, and check for overfitting before selecting a
final configuration.

**Rules followed throughout this notebook:**
- All tuning is done with cross-validation on the training set only.
- The held-out test set (`X_test`, `y_test`) is touched exactly once, at the
  very end, purely to report final validation numbers — never inside a
  search's `scoring`.
- Every experiment that matters is written to `experiment_log` and exported
  to `Experiment_Log.csv`.

> **Note on the dataset/models below:** this notebook was generated without
> access to your actual Day 3 notebook, so it uses the sklearn breast-cancer
> dataset as a runnable stand-in, and assumes **RandomForest** and
> **XGBoost** as the top-2 Day-3 candidates. Every place you need to swap in
> your real Day 3 outputs is marked `# PLACEHOLDER`. Swap those cells for
> your actual `X_train/X_test/y_train/y_test` (same split/random_state as
> Day 3) and your actual top-2 model classes, and the rest of the notebook
> runs unchanged.


## 1. Setup

In [1]:
import time
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load Day 3 data (PLACEHOLDER)

Replace this cell with the exact `X_train / X_test / y_train / y_test` used
in Day 3 (same feature engineering, same `random_state`), so the baseline
and candidate numbers stay comparable across days.

`PRIMARY_KPI` should be set to whatever your team defined as the primary
success metric in Day 3 (e.g. `f1`, `roc_auc`, `recall`).

In [2]:
# PLACEHOLDER: swap for your real Day 3 train/test split
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

PRIMARY_KPI = "f1"  # <-- set to Day 3's primary KPI

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)


(455, 30) (114, 30)


## 3. Experiment logging utilities

Every experiment we care about (baseline, untuned candidates, tuned
candidates) gets one row here: model, stage, hyperparameters, CV score,
held-out validation score, train score (for the overfitting check),
runtime, and a short observation. This becomes `Experiment_Log.csv` at
the end.

In [3]:
LOG_COLUMNS = [
    "Experiment_ID", "Date", "Model", "Stage", "Parameters",
    "CV_Score_Mean", "CV_Score_Std", "Validation_Score",
    "Train_Score", "Primary_KPI_Value", "Runtime_Sec", "Observation"
]
experiment_log = []
_exp_counter = 0


def log_experiment(model_name, stage, params, cv_mean, cv_std, val_score,
                    train_score, kpi_value, runtime_sec, observation):
    global _exp_counter
    _exp_counter += 1
    experiment_log.append({
        "Experiment_ID": f"EXP{_exp_counter:03d}",
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Model": model_name,
        "Stage": stage,
        "Parameters": json.dumps(params, default=str),
        "CV_Score_Mean": round(cv_mean, 4) if cv_mean is not None else None,
        "CV_Score_Std": round(cv_std, 4) if cv_std is not None else None,
        "Validation_Score": round(val_score, 4) if val_score is not None else None,
        "Train_Score": round(train_score, 4) if train_score is not None else None,
        "Primary_KPI_Value": round(kpi_value, 4) if kpi_value is not None else None,
        "Runtime_Sec": round(runtime_sec, 3),
        "Observation": observation,
    })


def evaluate_holdout(model, X_tr, y_tr, X_te, y_te):
    """Fit on train, report train F1 (for overfitting check) and
    test/validation F1, accuracy, AUC. This is the ONLY place the test set
    is touched, and only for reporting — never for model selection."""
    model.fit(X_tr, y_tr)
    train_pred = model.predict(X_tr)
    test_pred = model.predict(X_te)
    train_f1 = f1_score(y_tr, train_pred)
    test_f1 = f1_score(y_te, test_pred)
    test_acc = accuracy_score(y_te, test_pred)
    test_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    return train_f1, test_f1, test_acc, test_auc


## 4. Baseline model

A simple, unoptimized model (logistic regression) gives a reference point:
any tuned candidate that doesn't clearly beat this isn't worth the added
complexity.

In [4]:
baseline_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

t0 = time.time()
cv_res = cross_validate(baseline_pipe, X_train, y_train, cv=cv_strategy,
                         scoring="f1", return_train_score=True)
runtime = time.time() - t0
train_f1, test_f1, test_acc, test_auc = evaluate_holdout(
    baseline_pipe, X_train, y_train, X_test, y_test
)
log_experiment(
    "LogisticRegression", "Baseline", {"max_iter": 1000},
    cv_res["test_score"].mean(), cv_res["test_score"].std(),
    test_f1, train_f1, test_f1, runtime,
    "Simple linear baseline for reference."
)
print(f"Baseline CV F1: {cv_res['test_score'].mean():.4f} | Test F1: {test_f1:.4f}")


Baseline CV F1: 0.9825 | Test F1: 0.9861


## 5. Top 2 candidates from Day 3 — untuned

**PLACEHOLDER:** replace `RandomForestClassifier` / `XGBClassifier` with
your actual top-2 model classes from Day 3 if different. These are run
with library defaults first, so we can later show tuning's actual lift
over "just picking the model."</br>

In [5]:
# PLACEHOLDER: swap in Day 3's real top-2 model classes if different
rf_default = RandomForestClassifier(random_state=RANDOM_STATE)
xgb_default = XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss")

RF_KEY_PARAMS = ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features"]
XGB_KEY_PARAMS = ["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree", "min_child_weight", "gamma"]

for name, model, key_params in [
    ("RandomForest", rf_default, RF_KEY_PARAMS),
    ("XGBoost", xgb_default, XGB_KEY_PARAMS),
]:
    t0 = time.time()
    cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                             scoring="f1", return_train_score=True)
    runtime = time.time() - t0
    train_f1, test_f1, test_acc, test_auc = evaluate_holdout(
        model, X_train, y_train, X_test, y_test
    )
    all_params = model.get_params()
    shown_params = {k: all_params.get(k) for k in key_params}
    log_experiment(
        name, "Untuned Candidate", shown_params,
        cv_res["test_score"].mean(), cv_res["test_score"].std(),
        test_f1, train_f1, test_f1, runtime,
        "Default hyperparameters, no tuning applied."
    )
    print(f"{name} (untuned) CV F1: {cv_res['test_score'].mean():.4f} | Test F1: {test_f1:.4f} | Train F1: {train_f1:.4f}")


RandomForest (untuned) CV F1: 0.9699 | Test F1: 0.9655 | Train F1: 1.0000


XGBoost (untuned) CV F1: 0.9738 | Test F1: 0.9660 | Train F1: 1.0000


## 6. Most influential hyperparameters & search space

**RandomForest** — the parameters with the largest effect on bias/variance
for a tree ensemble:
- `n_estimators` — more trees stabilize the ensemble (diminishing returns, cost trade-off)
- `max_depth` — directly controls per-tree overfitting
- `min_samples_split` / `min_samples_leaf` — regularize tree growth, curb variance
- `max_features` — decorrelates trees, standard RF lever

**XGBoost** — the parameters that dominate boosted-tree behavior:
- `n_estimators` × `learning_rate` — jointly control fit strength vs. overfitting
- `max_depth` / `min_child_weight` — per-tree complexity and split sensitivity
- `subsample` / `colsample_bytree` — stochastic regularization
- `gamma` — minimum loss reduction to split (extra regularization)

The ranges below are centered on commonly effective values for
small-to-medium tabular datasets, wide enough to let the search meaningfully
explore, but not so wide that most draws are obviously bad (wasted budget).

In [6]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 4, 6, 8, 10, 12],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

xgb_param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
}


## 7. Tuning method: `RandomizedSearchCV`

`RandomizedSearchCV` is used instead of exhaustive `GridSearchCV` because
the combined search space (5 × 6 × 3 × 3 × 3 for RF, 4 × 5 × 5 × 3 × 3 × 3 × 3
for XGBoost) is large enough that a full grid would cost far more compute
for very little extra benefit over a well-sized random sample — a standard,
supervisor-approvable trade-off for this size of search space.

**Critical:** `scoring` is evaluated purely via `cv_strategy` on
`X_train / y_train`. `X_test / y_test` are never passed into either
search — the test set is not used for tuning.

In [7]:
tuned_models = {}

t0 = time.time()
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=rf_param_dist,
    n_iter=20, scoring="f1", cv=cv_strategy,
    random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
rf_search.fit(X_train, y_train)   # <-- train set + CV only
rf_runtime = time.time() - t0
tuned_models["RandomForest"] = rf_search

t0 = time.time()
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
    param_distributions=xgb_param_dist,
    n_iter=30, scoring="f1", cv=cv_strategy,
    random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
xgb_search.fit(X_train, y_train)  # <-- train set + CV only
xgb_runtime = time.time() - t0
tuned_models["XGBoost"] = xgb_search

print("RF best CV F1:", rf_search.best_score_)
print("XGB best CV F1:", xgb_search.best_score_)


RF best CV F1: 0.9701813114156721
XGB best CV F1: 0.9809282403332746


## 8. Log tuned candidates and evaluate on the held-out test set

This is the *only* point the test set is used — purely to report a final,
untouched validation number for the already-selected best configuration
from each search. It plays no role in choosing hyperparameters.

In [8]:
for name, search, runtime in [
    ("RandomForest", rf_search, rf_runtime),
    ("XGBoost", xgb_search, xgb_runtime),
]:
    best_model = search.best_estimator_
    train_f1, test_f1, test_acc, test_auc = evaluate_holdout(
        best_model, X_train, y_train, X_test, y_test
    )
    cv_mean = search.cv_results_["mean_test_score"][search.best_index_]
    cv_std = search.cv_results_["std_test_score"][search.best_index_]
    log_experiment(
        name, "Tuned Candidate", search.best_params_,
        cv_mean, cv_std, test_f1, train_f1, test_f1, runtime,
        f"Best of {search.n_iter} RandomizedSearchCV trials (5-fold CV)."
    )
    print(f"{name} (tuned) CV F1: {cv_mean:.4f} | Test F1: {test_f1:.4f} | Train F1: {train_f1:.4f}")


RandomForest (tuned) CV F1: 0.9702 | Test F1: 0.9583 | Train F1: 0.9948
XGBoost (tuned) CV F1: 0.9809 | Test F1: 0.9730 | Train F1: 1.0000


## 9. Compare: Baseline vs. Untuned vs. Tuned

A tidy view of the experiment log so far, sorted for readability.

In [9]:
log_df = pd.DataFrame(experiment_log, columns=LOG_COLUMNS)
comparison_view = log_df[[
    "Experiment_ID", "Model", "Stage", "CV_Score_Mean",
    "Validation_Score", "Train_Score", "Runtime_Sec"
]]
comparison_view


,Experiment_ID,Model,Stage,CV_Score_Mean,Validation_Score,Train_Score,Runtime_Sec
0,EXP001,LogisticRegression,Baseline,0.9825,0.9861,0.9913,0.061
1,EXP002,RandomForest,Untuned Candidate,0.9699,0.9655,1.0000,0.800
2,EXP003,XGBoost,Untuned Candidate,0.9738,0.9660,1.0000,0.292
3,EXP004,RandomForest,Tuned Candidate,0.9702,0.9583,0.9948,67.786
4,EXP005,XGBoost,Tuned Candidate,0.9809,0.9730,1.0000,14.486


## 10. Overfitting check

Comparing training performance to validation/CV performance for each tuned
candidate. A large gap (train ≫ validation) signals overfitting; a small,
consistent gap is expected and healthy.

In [10]:
overfit_rows = []
for name in ["RandomForest", "XGBoost"]:
    best_model = tuned_models[name].best_estimator_
    train_f1, test_f1, test_acc, test_auc = evaluate_holdout(
        best_model, X_train, y_train, X_test, y_test
    )
    gap = train_f1 - test_f1
    overfit_rows.append({
        "Model": name,
        "Train_F1": round(train_f1, 4),
        "Validation_F1": round(test_f1, 4),
        "Gap": round(gap, 4),
        "Flag": "Possible overfitting" if gap > 0.05 else "Acceptable gap",
    })

overfit_df = pd.DataFrame(overfit_rows)
overfit_df


,Model,Train_F1,Validation_F1,Gap,Flag
0,RandomForest,0.9948,0.9583,0.0364,Acceptable gap
1,XGBoost,1.0000,0.9730,0.0270,Acceptable gap


## 11. Select and save the best configuration

In [11]:
best_name = max(tuned_models, key=lambda n: tuned_models[n].best_score_)
best_search = tuned_models[best_name]

print("Selected model:", best_name)
print("Best hyperparameters:", best_search.best_params_)
print("Best CV F1:", best_search.best_score_)

joblib.dump(best_search.best_estimator_, "best_model_pipeline.joblib")

with open("best_config.json", "w") as f:
    json.dump({
        "model": best_name,
        "parameters": best_search.best_params_,
        "cv_f1_score": best_search.best_score_,
    }, f, indent=2)

print("Saved: best_model_pipeline.joblib, best_config.json")


Selected model: XGBoost
Best hyperparameters: {'subsample': 0.6, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 1.0}
Best CV F1: 0.9809282403332746
Saved: best_model_pipeline.joblib, best_config.json


## 12. Export the experiment log

In [12]:
log_df.to_csv("Experiment_Log.csv", index=False)
log_df


,Experiment_ID,Date,Model,Stage,Parameters,CV_Score_Mean,CV_Score_Std,Validation_Score,Train_Score,Primary_KPI_Value,Runtime_Sec,Observation
0,EXP001,2026-09-03 07:28,LogisticRegression,Baseline,"{""max_iter"": 1000}",0.9825,0.0078,0.9861,0.9913,0.9861,0.061,Simple linear baseline for reference.
1,EXP002,2026-09-03 07:28,RandomForest,Untuned Candidate,"{""n_estimators"": 100, ""max_depth"": null, ""min_...",0.9699,0.0146,0.9655,1.0000,0.9655,0.800,"Default hyperparameters, no tuning applied."
2,EXP003,2026-09-03 07:28,XGBoost,Untuned Candidate,"{""n_estimators"": null, ""max_depth"": null, ""lea...",0.9738,0.0077,0.9660,1.0000,0.9660,0.292,"Default hyperparameters, no tuning applied."
3,EXP004,2026-09-03 07:29,RandomForest,Tuned Candidate,"{""n_estimators"": 200, ""min_samples_split"": 5, ...",0.9702,0.0145,0.9583,0.9948,0.9583,67.786,Best of 20 RandomizedSearchCV trials (5-fold CV).
4,EXP005,2026-09-03 07:29,XGBoost,Tuned Candidate,"{""subsample"": 0.6, ""n_estimators"": 200, ""min_c...",0.9809,0.0149,0.9730,1.0000,0.9730,14.486,Best of 30 RandomizedSearchCV trials (5-fold CV).


## 13. Conclusion — was tuning worth it?

Fill this in after running with your real Day 3 data (the text below
reflects the placeholder dataset's results as an example of the reasoning
expected here):

- Compare each model's **Untuned Candidate** CV/validation score to its
  **Tuned Candidate** score in the table from step 9.
- A meaningful improvement is a validation/CV score gain that's larger than
  the run-to-run CV standard deviation (`CV_Score_Std`) — anything smaller
  is noise, not signal.
- Note the overfitting gap from step 10 alongside the improvement: a tuned
  model that gains 1% CV score but doubles its train/validation gap is a
  worse trade than a tuned model with a similar gain and a stable gap.
- State explicitly which model was selected as final and why (best
  validated score, acceptable overfitting gap, reasonable runtime).
